### Filter gold standard list from manual literature review
### Julian Moran
### 2026-04-01

In [1]:
import logging
import os
import sys

from dotenv import load_dotenv
from pathlib import Path

import polars as pl

# Env
pl.Config.set_tbl_rows(10)
load_dotenv("../.env")
INSTALL_PATH = os.environ["INSTALL_PATH"]
sys.path.append(str(Path(INSTALL_PATH).resolve()))

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)
logger = logging.getLogger(__name__)

### Goals

1. Filter for `Include = TRUE`

<br>

2.  Make pie plot of bacterial systems

In [8]:
# ────────────────────────────────────────────────────────────
#      Args
# ────────────────────────────────────────────────────────────

DATA_FILE = f"{INSTALL_PATH}/lit_review_manual/___bacterial_immuneSystem_goldStandard_manual.tsv"
OUT_DIR = f"{INSTALL_PATH}/lit_review_manual/"

In [13]:
# ────────────────────────────────────────────────────────────
#      In
# ────────────────────────────────────────────────────────────

df_goldStandard = pl.read_csv(
    DATA_FILE,
    separator="\t",
    has_header=True,
    encoding="latin1",
    null_values=["NA", "Not assessed"]
)

df_goldStandard

PMID,Paper_title,Bacterial_gene_name,Bacterial_species,Bacterial_UniProt_accession,Bacterial_AA_sequence,Human_gene_name,Human_UniProt_accession,Human_AA_sequence,Similiarity_to_hs_note,Include,Inclusion_reason,Bacterial_system,Function_in_bacteria,Functional_assay,Cloned_into_organism,Did_structural_validation,Screening_source,Note
i64,str,str,str,str,str,str,str,str,str,bool,str,str,str,str,str,str,str,str
12110595,"""tadA, an essential tRNA-specif…","""tadA""","""Escherichia coli""","""P68398""",null,null,null,null,null,false,"""No obvious immune function""",null,"""tRNA binding; tRNA deamination""","""Not assesed""","""Not assesed""","""Not assesed""","""C Trost""","""34% sequence-similar to yeast …"
30787435,"""Bacterial cGAS-like enzymes sy…","""dncV""","""Vibrio cholerae""","""Q9KVG7""",null,"""CGAS""","""Q8N884""","""MQPWHGKAMQRASEAGATAPKASARNARGA…","""Unquantified loose sequence si…",false,"""Functional assay, homology to …","""CBASS""","""abortive infection; intracellu…","""product radiolabelling""","""HEK293T""","""TRUE""","""C Trost""","""protein full name: Cyclic GMP-…"
30787435,"""Bacterial cGAS-like enzymes sy…","""cdnE""","""Escherichia coli""","""Q6XGD5""","""MGIPESQLDTWSHQGSIAQSASTYSIIKNA…","""CGAS""","""Q8N884""","""MQPWHGKAMQRASEAGATAPKASARNARGA…","""Unquantified loose sequence si…",true,"""Functional assay, homology to …","""CBASS""","""abortive infection; intracellu…","""biochemical deconvolution; act…",null,"""TRUE""","""C Trost""","""protein full name: Cyclic UMP-…"
30787435,"""Bacterial cGAS-like enzymes sy…","""cdnE""","""Rhodothermus marinus""","""G2SLH8""","""MPVPESQLERWSHQGATTTAKKTHESIRAA…","""CGAS""","""Q8N884""","""MQPWHGKAMQRASEAGATAPKASARNARGA…","""Unquantified loose sequence si…",true,"""Functional assay, homology to …","""CBASS""","""abortive infection; intracellu…","""product radiolabelling""",null,"""TRUE""","""C Trost""",null
31695182,"""The pan-immune system of bacte…","""Cas1""","""Escherichia coli""","""A0A1Q4PK74""",null,null,null,null,null,false,"""Review""","""CRISPR Cas""",null,"""Not assesed""","""Not assesed""","""Not assesed""","""C Trost""",null
31695182,"""The pan-immune system of bacte…","""Cas2""","""Escherichia coli""","""P45956""",null,null,null,null,null,false,"""Review""","""CRISPR Cas""",null,"""Not assesed""","""Not assesed""","""Not assesed""","""C Trost""",null
31695182,"""The pan-immune system of bacte…","""Cas9""","""Streptococcus pyogenes serotyp…","""Q99ZW2""",null,null,null,null,null,false,"""Review""","""CRISPR Cas""",null,"""Not assesed""","""Not assesed""","""Not assesed""","""C Trost""","""Cas9 is not endogenously expre…"
31695182,"""The pan-immune system of bacte…",null,null,null,null,null,null,null,null,false,"""Review""",null,null,null,null,null,"""C Trost""",null
31932165,"""HORMA Domain Proteins and a Tr…","""cdnC""","""Escherichia coli""","""D7Y2H2""",null,null,null,null,null,false,"""No human similarity""","""CBASS""","""abortive infection; intracellu…","""second-messenger synthesis ass…","""Escherichia coli""","""TRUE""","""C Trost""",null


In [17]:
# ────────────────────────────────────────────────────────────
#      Filter
# ────────────────────────────────────────────────────────────

df_filt = df_goldStandard.filter(
    (pl.col("Include") == True) &
    (~pl.col("Bacterial_gene_name").is_null())
)

df_filt.write_csv(
    f"{OUT_DIR}/bacterial_immSys_goldStnd_man_inclusionSet.tsv",
    separator="\t",
    include_header=True
)

unique_pairs = df_filt.select(
    ["Bacterial_UniProt_accession", "Human_UniProt_accession"]
).unique()

logger.info(f"Unique bacteria-human protein pairs: {len(unique_pairs)}")


df_filt

INFO:__main__:Unique bacteria-human protein pairs: 18


PMID,Paper_title,Bacterial_gene_name,Bacterial_species,Bacterial_UniProt_accession,Bacterial_AA_sequence,Human_gene_name,Human_UniProt_accession,Human_AA_sequence,Similiarity_to_hs_note,Include,Inclusion_reason,Bacterial_system,Function_in_bacteria,Functional_assay,Cloned_into_organism,Did_structural_validation,Screening_source,Note
i64,str,str,str,str,str,str,str,str,str,bool,str,str,str,str,str,str,str,str
30787435,"""Bacterial cGAS-like enzymes sy…","""cdnE""","""Escherichia coli""","""Q6XGD5""","""MGIPESQLDTWSHQGSIAQSASTYSIIKNA…","""CGAS""","""Q8N884""","""MQPWHGKAMQRASEAGATAPKASARNARGA…","""Unquantified loose sequence si…",true,"""Functional assay, homology to …","""CBASS""","""abortive infection; intracellu…","""biochemical deconvolution; act…",null,"""TRUE""","""C Trost""","""protein full name: Cyclic UMP-…"
30787435,"""Bacterial cGAS-like enzymes sy…","""cdnE""","""Rhodothermus marinus""","""G2SLH8""","""MPVPESQLERWSHQGATTTAKKTHESIRAA…","""CGAS""","""Q8N884""","""MQPWHGKAMQRASEAGATAPKASARNARGA…","""Unquantified loose sequence si…",true,"""Functional assay, homology to …","""CBASS""","""abortive infection; intracellu…","""product radiolabelling""",null,"""TRUE""","""C Trost""",null
40705877,"""A human homolog of SIR2 antiph…","""ThsA""","""Bacillus cereus""","""J8G6Z1""","""MKMNPIVELFIKDFTKEVMEENAAIFAGAG…","""FAM118B""","""Q9BPY3""","""MASTGSQASDIDEIFGFFNDGEPPTKKPRK…","""Human protein AKA ""SirAL"" by a…",true,"""Human homology""","""SIR2 antiphage; Thoeris""","""antiphage; NAD+ hydrolysis; NA…",null,null,null,"""ChatGPT""","""AKA ""Thoeris A"" by authors; an…"
32499527,"""Structural and functional evid…","""ThsA""","""Bacillus cereus""","""J8G6Z1""","""MKMNPIVELFIKDFTKEVMEENAAIFAGAG…","""FAM118B""","""Q9BPY3""","""MASTGSQASDIDEIFGFFNDGEPPTKKPRK…",null,true,"""Functional assays""",null,null,"""analytical size-exclusion chro…","""Escherichia coli""","""TRUE""","""ChatGPT""","""""Fig. 2 Crystal structure of T…"
40705877,"""A human homolog of SIR2 antiph…","""ThsA""","""Bacillus cereus""","""J8G6Z1""","""MKMNPIVELFIKDFTKEVMEENAAIFAGAG…","""SIRT1""","""Q96EB6""","""MADEAALALQPGGSPSAAGADREAASSPAG…",null,true,"""Human homology""","""SIR2 antiphage; Thoeris""","""antiphage; NAD+ hydrolysis; NA…",null,null,null,"""ChatGPT""","""AKA ""Thoeris A"" by authors; an…"
32499527,"""Structural and functional evid…","""ThsA""","""Bacillus cereus""","""J8G6Z1""","""MKMNPIVELFIKDFTKEVMEENAAIFAGAG…","""SIRT1""","""Q96EB6""","""MADEAALALQPGGSPSAAGADREAASSPAG…",null,true,"""Functional assays""",null,null,"""analytical size-exclusion chro…","""Escherichia coli""","""TRUE""","""ChatGPT""","""""Fig. 2 Crystal structure of T…"
40705877,"""A human homolog of SIR2 antiph…","""ThsA""","""Bacillus cereus""","""J8G6Z1""","""MKMNPIVELFIKDFTKEVMEENAAIFAGAG…","""SIRT2""","""Q8IXJ6""","""MAEPDPSHPLETQAGKVQEAQDSDSDSEGG…",null,true,"""Human homology""","""SIR2 antiphage; Thoeris""","""antiphage; NAD+ hydrolysis; NA…",null,null,null,"""ChatGPT""","""AKA ""Thoeris A"" by authors; an…"
32499527,"""Structural and functional evid…","""ThsA""","""Bacillus cereus""","""J8G6Z1""","""MKMNPIVELFIKDFTKEVMEENAAIFAGAG…","""SIRT2""","""Q8IXJ6""","""MAEPDPSHPLETQAGKVQEAQDSDSDSEGG…",null,true,"""Functional assays""",null,null,"""analytical size-exclusion chro…","""Escherichia coli""","""TRUE""","""ChatGPT""","""""Fig. 2 Crystal structure of T…"
40705877,"""A human homolog of SIR2 antiph…","""ThsA""","""Bacillus cereus""","""J8G6Z1""","""MKMNPIVELFIKDFTKEVMEENAAIFAGAG…","""SIRT4""","""Q9Y6E7""","""MKMSFALTFRSAKGRWIANPSQPCSKASIG…",null,true,"""Human homology""","""SIR2 antiphage; Thoeris""","""antiphage; NAD+ hydrolysis; NA…",null,null,null,"""ChatGPT""","""AKA ""Thoeris A"" by authors; an…"


In [11]:
pl.Config.set_tbl_rows(30)

df_filt[["PMID", "Bacterial_gene_name", "Bacterial_Uniprot", "Human_gene_name"]]

PMID,Bacterial_gene_name,Human_gene_name
i64,str,str
30787435,"""cdnE""","""CGAS"""
30787435,"""cdnE""","""CGAS"""
40705877,"""ThsA""","""FAM118B"""
32499527,"""ThsA""","""FAM118B"""
40705877,"""ThsA""","""SIRT1"""
32499527,"""ThsA""","""SIRT1"""
40705877,"""ThsA""","""SIRT2"""
32499527,"""ThsA""","""SIRT2"""
40705877,"""ThsA""","""SIRT4"""
